In [1]:
# Imports
import pandas as pd
from random import randint
from src import *
from src.simulator import SIMULATOR
import numpy as np

def printAsMatrix(array, rows, cols):
    for i in range(rows):
        print(array[i * cols:(i + 1) * cols])

In [2]:
# --------------------------------------------
#               INIT & CONFIG
# --------------------------------------------
sim = SIMULATOR()

DEBUG = 1
MAX_ITER = 300000

# DISCO-CGRA Parameters
nRCs = 4
nElementsPerVWRSlice = 32
nColsCGRA = 1
nElemsPerSPMLine = 128

In [3]:
# --------------------------------------------
#               KERNEL CONFIGURATION
# --------------------------------------------
kernel_path = './kernels/mmul_v1_tiling_32x32/'
kernel_number = 1 
column_usage = [True, True] 
nInstrPerCol = 45
imem_add_start = 0 
srf_spm_addres = 0 
version="_2col"

# Block size 32x32
BLOCK_SIZE = 32

sim.kernel_config(column_usage, nInstrPerCol, imem_add_start, srf_spm_addres, kernel_number)

In [ ]:
# --------------------------------------------
#               DATA
# --------------------------------------------
data = np.load(kernel_path + "data/data.npz")

A = data["A"]
B = data["B"]
C = data["C"]
expected_res = data["D"]

ROWS_A = int(data["ROWS_A"])
COLS_A = int(data["COLS_A"])
COLS_B = int(data["COLS_B"])

B_t = ((B.reshape(COLS_A, COLS_B)).T).flatten()

print(f"Testing sizes A: {ROWS_A}x{COLS_A}, B: {COLS_A}x{COLS_B}, C: {ROWS_A}x{COLS_B}")


In [ ]:
# --------------------------------------------
#              COMPILE ASM TO HEX
# --------------------------------------------
sim.compileAsmToHex(kernel_path, kernel_number, version=version)

# --------------------------------------------
#          LOAD KERNEL INSTRUCTIONS
# --------------------------------------------

# This needs the hex instructions, if you don't provide them, generate then compiling the asm
sim.kernel_load(kernel_path, version=version + "_autogen", kernel_number=kernel_number)

In [6]:
# --------------------------------------------
#          FILL SPM
# --------------------------------------------
# Default SPM lines
srf_spm_line = 0
nLinesPerMatrix = 8

a_spm_line_ini = 1
b_spm_line_ini = a_spm_line_ini + nLinesPerMatrix
c_spm_line_ini = b_spm_line_ini + nLinesPerMatrix

# A
a_spm_line = a_spm_line_ini
a_row = 0
for r in range(ROWS_A // 4):
    line = [0 for _ in range(nElemsPerSPMLine)]
    for rr in range(4):
        line[nElementsPerVWRSlice*rr : nElementsPerVWRSlice*rr + COLS_A] = A[(a_row + rr)*COLS_A:(a_row + rr)*COLS_A + COLS_A].copy()
    sim.setSPMLine(a_spm_line, line.copy())
    a_spm_line += 1
    a_row+=4


# B_t
b_spm_line = b_spm_line_ini
b_t_row = 0
for c in range(COLS_B // 4):
    line = [0 for _ in range(nElemsPerSPMLine)]
    for rr in range(4):
        line[nElementsPerVWRSlice*rr : nElementsPerVWRSlice*rr + COLS_A] = B_t[(b_t_row + rr)*COLS_A:(b_t_row + rr)*COLS_A + COLS_A].copy()
    sim.setSPMLine(b_spm_line, line.copy())
    b_spm_line += 1
    b_t_row+=4

#C
c_spm_line = c_spm_line_ini
c_row = 0
for r in range(ROWS_A // 4):
    line = [0 for _ in range(nElemsPerSPMLine)]
    for rr in range(4):
        line[nElementsPerVWRSlice*rr : nElementsPerVWRSlice*rr + COLS_B] = C[(c_row + rr)*COLS_B:(c_row + rr)*COLS_B + COLS_B].copy()
    sim.setSPMLine(c_spm_line, line.copy())
    c_spm_line += 1
    c_row+=4

In [ ]:
# SRF
# --------------------------------------------
# SRF0 = SPMA
# SRF1 = SPMB
# SRF2 = SPMC
# SRF3 = itL1 (row blocking)
# SRF4 = itL2 (col blocking)
# SRF5 = itL3 (inner dimension)
# SRF6 = -
# SRF7 = -
# --------------------------------------------

# Default SRF values
srf = [0 for i in range(N_ELEMS_PER_VWR)]

nLinesPerRow = (ROWS_A // 4)//2

# Col 0
srf[0]  = a_spm_line_ini 
srf[1]  = b_spm_line_ini 
srf[2]  = c_spm_line_ini
srf[3]  = (ROWS_A // 4)//2 -1 # ROWS_A / BLOCK_SIZE=4 / N_COLS=2
srf[4]  = COLS_B // 4 -1 
srf[5]  = COLS_A -1 
srf[6]  = 0      # Not used
srf[7]  = 0      # Not used
# Col 1
srf[8]  = a_spm_line_ini + nLinesPerRow # Compute the next rows of C
srf[9]  = b_spm_line_ini
srf[10] = c_spm_line_ini + nLinesPerRow # Compute the next rows of C
srf[11] = (ROWS_A // 4)//2 -1
srf[12] = COLS_B // 4 -1 
srf[13] = COLS_A -1 
srf[14] = 0      # Not used
srf[15] = 0      # Not used

sim.setSPMLine(srf_spm_line, srf.copy())
print(f"SRF: {srf}")

In [ ]:
# --------------------------------------------
#               SIMULATE EXECUTION
# --------------------------------------------
show_lcu = []
show_srf = []
show_lsu = []
show_rcs = [[],[],[],[]]
show_mxcu = []
display_ops = [show_lcu, show_lsu, show_mxcu, show_rcs, show_srf]

# Launch kernel
sim.run(kernel_number, display_ops=display_ops, max_iter=MAX_ITER)



In [9]:
def getCFromSPM():
    c_spm_line = c_spm_line_ini
    C_out = []

    for _ in range(ROWS_A // 4):
        line = sim.getSPMLine(c_spm_line)
        for rr in range(4):
            row_real = line[rr * nElementsPerVWRSlice : rr * nElementsPerVWRSlice + COLS_B]
            C_out.extend(c_int32(x).value for x in row_real)
        c_spm_line += 1

    return C_out
# Get kernel results
disco_out = getCFromSPM()


In [ ]:
# Verify results
errors = 0
for i in range(len(expected_res)):
    if expected_res[i] != disco_out[i]:
        errors+=1
    
if errors == 0:
    print("The result is correct!")
else:
    print("Oops, something went wrong. There are " + str(errors) + " errors (out of " + str(len(expected_res)) + " elements).")
    print("DISCO out:")
    printAsMatrix(disco_out, ROWS_A, COLS_B)
    print("Expected result:")
    printAsMatrix(expected_res, ROWS_A, COLS_B)
    print("A:")
    printAsMatrix(A, ROWS_A, COLS_A)
    print("B_t:")
    printAsMatrix(B_t, COLS_B, COLS_A)
    print("C:")
    printAsMatrix(C, ROWS_A, COLS_B)